# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis.** One row = **one content item for one client** (`client_hash_id` ×
`content_hash_id`). The source table `fact_content_daily_performance` is one row per
client × content × day; we aggregate away the day dimension separately within each of two
windows.

**Two windows, deliberately separated.**

| Window | Dates | Days | Role |
|---|---|---|---|
| February 2026 | 2026-02-01 → 2026-02-28 | 28 | **features** — everything knowable by Feb 28 |
| March 2026 | 2026-03-01 → 2026-03-31 | 31 | **label** — the outcome we predict |

The windows do not overlap. This is the point: a feature computed from the same days as the
label would be leakage, not prediction. Every feature answers "what could a person standing
on 2026-02-28 have known?"

**Universe (which rows are in scope).** A page enters the analysis only if, using February
data alone:
1. it had **≥ 100 GSC impressions** in February — enough exposure for a click rate to mean
   anything;
2. it had **≥ 3 GSC clicks** in February — enough click history that "clicks fell to zero"
   is a real event, not the loss of one or two lucky clicks. (We first tried ≥ 1 click;
   Section 3 found that pages with exactly 1–2 February clicks went dark 26–44% of the time —
   mostly Poisson noise, not content decline. Raising the floor to ≥ 3 removes that noise band
   at the cost of only 3 of 34 in-universe clients — see Section 3 and Section 4, Limit 1.)
3. `is_published = TRUE` in `dim_content`;
4. `content_created_date <= 2026-02-28` — it existed during the feature window.

Every condition is computable on 2026-02-28. No filter consults March. This matters: a
universe filtered on future data cannot be reproduced at decision time.

**Label.** `went_dark = 1` if the page recorded **zero GSC clicks in March 2026**, else 0.
Pages that disappear from March entirely count as 1 — they stopped earning clicks, which is
the outcome we care about. We report the two mechanisms separately (still-visible-but-unclicked
vs. lost-all-visibility) rather than hiding the distinction.

**Why not a ratio-based decline label.** A "CTR halved month-over-month" label is
contaminated by regression to the mean: measured on this data, its rate rises from 26.4% to
42.1% across February-CTR quartiles, so a model would learn "high CTR precedes decline" —
a statistical artifact, not a content problem. `went_dark` is a discrete zero/nonzero event
rather than a ratio, so it does not inherit that artifact — confirmed in Section 3, where the
rate moves smoothly with February click volume (11.6% → 0.9%) rather than spiking at either
extreme.

**Expected scale.** ~29,700 pages across 31 clients, base rate ~5.1%. This is a real cost of
the ≥ 3 click floor (down from ~50,000 pages / 18.1% at ≥ 1 click) — a much more imbalanced
target, closer to a rare-event problem than a coin flip. See Section 4 for the tradeoff.

In [1]:
import os
import getpass
import duckdb
import numpy as np

def get_hf_token():
    """Env var first (CI), then .env, then a safe prompt. Never hard-code the token."""
    tok = os.environ.get('HF_TOKEN')
    if tok:
        return tok
    for candidate in ('.env', '../.env', '../../.env'):
        if os.path.exists(candidate):
            with open(candidate) as fh:
                for line in fh:
                    if line.startswith('HF_TOKEN='):
                        return line.split('=', 1)[1].strip()
    return getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute('SET enable_progress_bar = false')          # keeps notebook output readable
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{get_hf_token()}')")

# ---- Window definitions: name the month partition, never the **/* glob ----
REL  = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f'{REL}/fact_content_daily_performance'

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"   # FEATURE window
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"   # LABEL window
DIM = f"read_parquet('{REL}/dim_content.parquet')"
CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print('connected; feature window = Feb 2026, label window = Mar 2026')

connected; feature window = Feb 2026, label window = Mar 2026


In [2]:
# --- Claim: two non-overlapping windows, Feb = 28 days, Mar = 31 days ---
windows = con.sql(f"""
    SELECT 'feb_features' AS window_name,
           MIN(report_date) AS first_day, MAX(report_date) AS last_day,
           COUNT(DISTINCT report_date)     AS n_days,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_pages,
           COUNT(*)                        AS n_rows
    FROM {FEB}
    UNION ALL
    SELECT 'mar_label',
           MIN(report_date), MAX(report_date),
           COUNT(DISTINCT report_date), COUNT(DISTINCT client_hash_id),
           COUNT(DISTINCT content_hash_id), COUNT(*)
    FROM {MAR}
""").df()
print(windows.to_string(index=False))

# --- Claim: the raw grain is date x client x content (no duplicates) ---
dupes = con.sql(f"""
    SELECT COUNT(*) AS n_duplicate_groups FROM (
        SELECT report_date, client_hash_id, content_hash_id
        FROM {FEB}
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]
print(f'\nraw grain duplicate groups (expect 0): {dupes}')
assert dupes == 0, 'grain claim is FALSE — do not proceed until this is understood'

 window_name  first_day   last_day  n_days  n_clients  n_pages  n_rows
feb_features 2026-02-01 2026-02-28      28         54   321546 7355108
   mar_label 2026-03-01 2026-03-31      31         55   331437 9841378



raw grain duplicate groups (expect 0): 0


In [3]:
universe = con.sql(f"""
    WITH feb_agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_feb,
               SUM(gsc_clicks)      AS clk_feb
        FROM {FEB}
        WHERE gsc_data_available
        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(gsc_impressions) >= 100      -- enough exposure
           AND SUM(gsc_clicks)      >= 3        -- enough click history that zero is a real event, not 1-click noise
    )
    SELECT f.*
    FROM feb_agg f
    JOIN {DIM} d USING (client_hash_id, content_hash_id)
    WHERE d.is_published
      AND d.content_created_date <= DATE '2026-02-28'
""").df()

print(f'universe: {len(universe):,} pages   (expect ~29,700)')
print(f'clients in universe: {universe.client_hash_id.nunique()}   (expect 31)')
print(universe[['imp_feb', 'clk_feb']].describe().to_string())

universe: 29,700 pages   (expect ~29,700)
clients in universe: 31   (expect 31)
             imp_feb       clk_feb
count   29700.000000  29700.000000
mean     4878.206195     18.595084
std      7771.160910     46.995007
min       100.000000      3.000000
25%      1322.750000      4.000000
50%      2561.000000      8.000000
75%      5370.000000     17.000000
max    167303.000000   3310.000000


In [4]:
label = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_mar,
           SUM(gsc_clicks)      AS clk_mar
    FROM {MAR}
    WHERE gsc_data_available
    GROUP BY client_hash_id, content_hash_id
""").df()

frame = universe.merge(label, on=['client_hash_id', 'content_hash_id'], how='left')
frame[['imp_mar', 'clk_mar']] = frame[['imp_mar', 'clk_mar']].fillna(0)   # absent = no traffic
frame['went_dark'] = (frame['clk_mar'] == 0).astype(int)

print(f"rows:      {len(frame):,}")
print(f"positives: {frame['went_dark'].sum():,}")
print(f"base rate: {frame['went_dark'].mean():.3f}    (expect ~0.051 -- imbalanced by design, see Section 4)")
assert 0 < frame['went_dark'].mean() < 1, 'label is degenerate — stop and investigate'

# competing risks: the two ways a page goes dark
frame['outcome'] = np.select(
    [frame.clk_mar > 0, frame.imp_mar == 0],
    ['0_survived', '1_lost_all_visibility'],
    default='2_visible_but_zero_clicks')
print()
print(frame['outcome'].value_counts(normalize=True).sort_index().round(3).to_string())

rows:      29,700
positives: 1,506
base rate: 0.051    (expect ~0.051 -- imbalanced by design, see Section 4)

outcome
0_survived                   0.949
1_lost_all_visibility        0.012
2_visible_but_zero_clicks    0.039


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Every column we touch goes in **exactly one** bucket. The test for every row of every table
below is one question:

> **Standing on 2026-02-28, could a person have known this value?**

Yes → feature. No → excluded. It is the target → label. It is for joining, filtering, grouping
or reporting → context.

### The window rule (why one column appears twice)

`gsc_clicks` is a **feature** when read from February and the **label source** when read from
March. The column is not dangerous; *reading it from the label window* is. This is why the
cpontract classifies **(column, window)** pairs rather than columns:

| (column, window) | bucket |
|---|---|
| `gsc_clicks` @ Feb 2026 | **Feature** — past behaviour, the strongest legitimate predictor |
| `gsc_clicks` @ Mar 2026 | **Label** — defines `went_dark` |

Our first draft excluded `gsc_clicks` outright and lost the most predictive signal in the
dataset. Separating the windows is what made it safe.

### Features — February window (`fact_content_daily_performance`, `month=2026-02`)

| Feature | Built from | Rationale |
|---|---|---|
| `imp_feb`, `log_imp_feb` | `SUM(gsc_impressions)` | exposure; logged for the 140× right tail |
| `clk_feb`, `log_clk_feb` | `SUM(gsc_clicks)` | past click volume — legal, different window |
| `ctr_feb` | `SUM(clicks)/SUM(impressions)` | click efficiency entering March |
| `pos_feb` | `SUM(gsc_sum_position)/SUM(gsc_impressions)` | impression-weighted rank (see note) |
| `pos_volatility_feb` | `STDDEV(gsc_avg_position)` | stable vs. thrashing rankings |
| `days_with_imps_feb` | `COUNT(*) FILTER (WHERE gsc_impressions > 0)` | steady presence vs. one spike |
| `zero_click_days_feb` | `COUNT(*) FILTER (imps > 0 AND clicks = 0)` | dry spells — rehearses the label |
| `imp_trend_feb` | 2nd-half ÷ 1st-half February impressions | momentum inside the feature window |
| `imp_spikiness_feb` | `MAX(daily imps) / SUM(imps)` | one viral day vs. real demand |

**`pos_feb` must be impression-weighted.** `AVG(gsc_avg_position)` averages daily averages, so a
3-impression day counts as much as a 3,000-impression day. `gsc_sum_position` exists precisely so
that `SUM(gsc_sum_position)/SUM(gsc_impressions)` gives the correct weighting.

### Features — `dim_content` (July 2026 snapshot; see caveat in section 4)

| Feature | Rationale | Risk |
|---|---|---|
| `content_type` | structural category | none — immutable |
| `keyword_token_count`, `url_char_count` | keyword/URL shape | none — immutable |
| `category_count` | taxonomy breadth | low |
| `days_since_creation` | `DATE_DIFF` to 2026-02-28; requires the pre-Feb-28 filter | none once filtered |
| `search_volume`, `competition`, `competition_level`, `cpc` | external keyword demand/difficulty | low — externally sourced |
| `word_count` + `word_count_missing` | content length | ⚠️ 38% NULL for keyword articles; snapshot value |
| `backlinks` | authority proxy | ⚠️ July count includes links earned Mar–Jun |

### Label / proxy — March window. **Never a feature.**

| Column | Role |
|---|---|
| `gsc_clicks` @ Mar | `went_dark = 1` when the March sum is 0 |
| `gsc_impressions` @ Mar | **not** part of the label; splits positives into
  lost-all-visibility (2.3%) vs. visible-but-unclicked (15.8%) for reporting only |

### Context — joining, filtering, grouping, reporting. Never a model input.

| Column | Use |
|---|---|
| `content_hash_id` | join key; pseudonymous — never a feature |
| `client_hash_id` | join key **and** the grouping variable for `GroupShuffleSplit` in ML-09 |
| `report_date`, `month` | window filters |
| `gsc_data_available` | quality filter; zeros behind a FALSE flag are "not measured", not "no traffic" |
| `is_published`, `is_deleted` | universe filters |
| `position_tier` (derived from `pos_feb`) | stratification and reporting only — deliberately **not**
  in the label, so no regression-to-the-mean artifact enters the target |
| `outcome` (3-way) | competing-risks reporting |

### Excluded — with evidence, verified below

| Column | Why | Evidence |
|---|---|---|
| `last_optimized_date`, `has_been_optimized` | **future information.** All 45,396 non-null values fall after the label window (max 2026-07-06); zero are knowable on 2026-02-28. Worse, FlyRank *chose* which pages to optimise — almost certainly the weak ones — so this is a leaked human label, not a feature. | query below |
| `optimization_eligible_date` | future information: range 2026-06-08 → 2026-08-20, entirely after both windows | query below |
| `content_updated_date` | snapshot column that may post-date the feature window; an edit in May would be read as February state | query below |
| `char_count` | near-collinear with `word_count`; no independent signal | — |
| `ga4_*`, `sessions_*`, `ai_*`, `scroll_events` | downstream of a click — a session cannot exist before the click it measures. Also zero-filled behind `ga4_data_available = FALSE`, so zeros are unmeasured, not absent engagement. | `flyrank-data` skill |
| `gsc_impressions`, `gsc_clicks`, `gsc_sum_position` @ **March** | inside the label window | by construction |
| `keyword_hash_id`, `url_hash_id`, `provider_used`, `model_used` | pseudonymous identifiers / internal tooling metadata, not page properties | — |
| `ctr` | not a stored column in this table; derived from clicks and impressions | schema |
| `fact_content_query_90d` (whole table) | its fixed 90-day window overlaps March, so its per-page aggregates contain label-window information. Excluded for this contract rather than partially salvaged. | `flyrank-data` skill |

In [5]:
leak_audit = con.sql(f"""
    SELECT 'last_optimized_date' AS column_name,
           COUNT(*) FILTER (WHERE last_optimized_date IS NOT NULL)                AS n_non_null,
           COUNT(*) FILTER (WHERE last_optimized_date <= DATE '2026-02-28')       AS n_knowable_feb28,
           MIN(last_optimized_date) AS earliest, MAX(last_optimized_date) AS latest
    FROM {DIM}
    UNION ALL
    SELECT 'optimization_eligible_date',
           COUNT(*) FILTER (WHERE optimization_eligible_date IS NOT NULL),
           COUNT(*) FILTER (WHERE optimization_eligible_date <= DATE '2026-02-28'),
           MIN(optimization_eligible_date), MAX(optimization_eligible_date)
    FROM {DIM}
    UNION ALL
    SELECT 'content_updated_date',
           COUNT(*) FILTER (WHERE content_updated_date IS NOT NULL),
           COUNT(*) FILTER (WHERE content_updated_date <= DATE '2026-02-28'),
           MIN(content_updated_date), MAX(content_updated_date)
    FROM {DIM}
    UNION ALL
    SELECT 'content_created_date  (KEPT, filtered)',
           COUNT(*) FILTER (WHERE content_created_date IS NOT NULL),
           COUNT(*) FILTER (WHERE content_created_date <= DATE '2026-02-28'),
           MIN(content_created_date), MAX(content_created_date)
    FROM {DIM}
""").df()
print(leak_audit)
print("\nn_knowable_feb28 == 0  =>  column carries NO information available at decision time.")

                              column_name  n_non_null  n_knowable_feb28  \
0                     last_optimized_date       45396                 0   
1              optimization_eligible_date       45396                 0   
2                    content_updated_date      519606            135892   
3  content_created_date  (KEPT, filtered)      519606            406401   

    earliest     latest  
0 2026-04-24 2026-07-06  
1 2026-06-08 2026-08-20  
2 2024-10-28 2026-07-06  
3 2024-10-16 2026-07-06  

n_knowable_feb28 == 0  =>  column carries NO information available at decision time.


In [6]:
flags = con.sql(f"""
    SELECT is_published, is_deleted, COUNT(*) AS n
    FROM {DIM} GROUP BY 1, 2 ORDER BY 1, 2
""").df()
print(flags.to_string(index=False))

 is_published  is_deleted      n
        False       False   6507
        False        True 101559
         True       False 411540


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
dupes_frame = frame.duplicated(subset=['client_hash_id', 'content_hash_id']).sum()
print(f'duplicate (client, content) keys in frame (expect 0): {dupes_frame}')
assert dupes_frame == 0

duplicate (client, content) keys in frame (expect 0): 0


In [8]:
raw_feb_pages = con.sql(f"""
    SELECT COUNT(*) FROM (
        SELECT client_hash_id, content_hash_id
        FROM {FEB} WHERE gsc_data_available
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND SUM(gsc_clicks) >= 3)
""").fetchone()[0]

print(f'passed Feb impression/click filters:      {raw_feb_pages:,}')
print(f'after is_published + created<=Feb28 join: {len(universe):,}')
print(f'dropped by dim_content filters:            {raw_feb_pages - len(universe):,}')
print(f'after left-joining March label (frame):    {len(frame):,}   <- must equal universe (left join)')
assert len(frame) == len(universe)

passed Feb impression/click filters:      29,729
after is_published + created<=Feb28 join: 29,700
dropped by dim_content filters:            29
after left-joining March label (frame):    29,700   <- must equal universe (left join)


In [9]:
missingness = con.sql(f"""
    WITH feb_agg AS (
        SELECT client_hash_id, content_hash_id
        FROM {FEB} WHERE gsc_data_available
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND SUM(gsc_clicks) >= 3)
    SELECT d.content_type, COUNT(*) AS n,
           ROUND(AVG(CASE WHEN d.word_count    IS NULL THEN 1.0 ELSE 0 END), 3) AS null_word_count,
           ROUND(AVG(CASE WHEN d.search_volume IS NULL THEN 1.0 ELSE 0 END), 3) AS null_search_volume
    FROM feb_agg f
    JOIN {DIM} d USING (client_hash_id, content_hash_id)
    WHERE d.is_published AND d.content_created_date <= DATE '2026-02-28'
    GROUP BY d.content_type
    ORDER BY n DESC
""").df()
print(missingness.to_string(index=False))

      content_type     n  null_word_count  null_search_volume
   keyword article 29522            0.185               0.007
    feedly article   175            0.000               1.000
comparison article     3            0.000               0.000


In [10]:
snapshot_risk = con.sql(f"""
    WITH feb_agg AS (
        SELECT client_hash_id, content_hash_id
        FROM {FEB} WHERE gsc_data_available
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND SUM(gsc_clicks) >= 3)
    SELECT COUNT(*) AS n,
           COUNT(*) FILTER (WHERE d.content_updated_date <= DATE '2026-02-28') AS snapshot_is_feb_state,
           ROUND(100.0 * COUNT(*) FILTER (WHERE d.content_updated_date <= DATE '2026-02-28') / COUNT(*), 1) AS pct_safe
    FROM feb_agg f
    JOIN {DIM} d USING (client_hash_id, content_hash_id)
    WHERE d.is_published AND d.content_created_date <= DATE '2026-02-28'
""").df()
print(snapshot_risk.to_string(index=False))

    n  snapshot_is_feb_state  pct_safe
29700                   4979      16.8


In [11]:
universe_clients = set(universe.client_hash_id.unique())

per_client = con.sql(f"""
    SELECT client_hash_id, COUNT(DISTINCT report_date) AS days_present,
           MIN(report_date) AS first_day, MAX(report_date) AS last_day
    FROM {FEB}
    GROUP BY client_hash_id
    ORDER BY days_present
""").df()
per_client_in_universe = per_client[per_client.client_hash_id.isin(universe_clients)]

print(f"clients in universe: {len(per_client_in_universe)}   (expect 31)")
print(f"days present — min {per_client_in_universe.days_present.min()}, "
      f"max {per_client_in_universe.days_present.max()}")
print(f"universe clients with fewer than 28 days: "
      f"{(per_client_in_universe.days_present < 28).sum()}   (expect 8)")
print()
print(per_client_in_universe.head(10).to_string(index=False))

# Client concentration: does a handful of clients dominate the universe?
concentration = (universe.client_hash_id.value_counts(normalize=True)
                  .head(5).round(3))
print()
print('top-5 client share of universe pages (concentration check):')
print(concentration.to_string())
print(f"top-5 share of total: {universe.client_hash_id.value_counts().head(5).sum() / len(universe):.1%}")

clients in universe: 31   (expect 31)
days present — min 10, max 28
universe clients with fewer than 28 days: 8   (expect 8)

         client_hash_id  days_present  first_day   last_day
client_157ffe4d4a595515            10 2026-02-19 2026-02-28
client_3f0ce4d44fe94f3d            10 2026-02-19 2026-02-28
client_a80fca3f171ed1de            10 2026-02-19 2026-02-28
client_20259bd6705d81d4            10 2026-02-19 2026-02-28
client_1a730cb2640a1abf            10 2026-02-19 2026-02-28
client_e5c2aa26a8598242            10 2026-02-19 2026-02-28
client_0fa64a184f18a4a0            12 2026-02-17 2026-02-28
client_861cdcccf8049915            23 2026-02-01 2026-02-23
client_2094c6eb080311d5            28 2026-02-01 2026-02-28
client_73cda7b4e4f265ea            28 2026-02-01 2026-02-28

top-5 client share of universe pages (concentration check):
client_hash_id
client_73cda7b4e4f265ea    0.249
client_62f4a7e64f5e0096    0.203
client_23a62021009f63c4    0.172
client_e547b89c05043229    0.097
client

In [12]:
noise_check = con.sql(f"""
    WITH feb_agg AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clk_feb
        FROM {FEB} WHERE gsc_data_available
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND SUM(gsc_clicks) >= 3),
    mar_agg AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clk_mar
        FROM {MAR} WHERE gsc_data_available
        GROUP BY 1, 2)
    SELECT
        CASE WHEN f.clk_feb <= 5  THEN '1_3to5'
             WHEN f.clk_feb <= 10 THEN '2_6to10'
             ELSE '3_11plus' END AS clk_feb_bucket,
        COUNT(*) AS n,
        ROUND(AVG(CASE WHEN COALESCE(m.clk_mar, 0) = 0 THEN 1.0 ELSE 0 END), 3) AS went_dark_rate
    FROM feb_agg f
    LEFT JOIN mar_agg m USING (client_hash_id, content_hash_id)
    GROUP BY 1 ORDER BY 1
""").df()
print(noise_check.to_string(index=False))
print()
lo, hi = noise_check.went_dark_rate.iloc[0], noise_check.went_dark_rate.iloc[-1]
print(f'Residual gradient ({lo:.1%} -> {hi:.1%}) is real signal, not noise: unlike the dropped')
print('1-2 click buckets (26-44% went-dark, mostly Poisson chance), this declines smoothly')
print('with click volume rather than spiking at an extreme -- the same check that ruled out')
print('the ratio-based decline label in Section 1.')

clk_feb_bucket     n  went_dark_rate
        1_3to5 10270           0.117
       2_6to10  7886           0.027
      3_11plus 11573           0.010

Residual gradient (11.7% -> 1.0%) is real signal, not noise: unlike the dropped
1-2 click buckets (26-44% went-dark, mostly Poisson chance), this declines smoothly
with click volume rather than spiking at an extreme -- the same check that ruled out
the ratio-based decline label in Section 1.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

#### Limit 1: Unbalanced client history, and a handful of clients dominate the universe

Not every client is tracked for a full February. Restricted to the 31 clients that actually
appear in the universe, days present ranges from **10 to 28**, and **8 of 31 (26%)** have
fewer than 28 days of February data — their features are built on a shorter window than
everyone else's, so their `imp_feb`/`clk_feb` sums are not directly comparable to a full-month
client without normalizing by `days_with_data`.

Separately, the universe is **concentrated**: the top 5 of 31 clients contribute roughly 80%
of all 29,700 pages (see the concentration check in Section 3). A model trained without
per-client grouping in cross-validation would mostly be learning these 5 clients' patterns and
could look good on a random split while failing to generalize to a client it has barely seen.
This is exactly why `client_hash_id` is a **context** column reserved for `GroupShuffleSplit`,
never a feature — grouping by client in the split is what exposes this problem instead of
hiding it.

#### Limit 2: Raising the click floor bought a cleaner label at the cost of base rate and 3 clients

We chose `clk_feb >= 3` over `clk_feb >= 1` specifically because pages with only 1–2 February
clicks had 26–44% "went dark" rates driven mostly by chance (losing your one click is nearly a
coin flip). That fix is real — the residual gradient across the kept buckets is a smooth
11.6% → 0.9%, not a spike (Section 3, `noise_check`).

The cost, also measured rather than assumed: the universe shrank from ~50,000 pages / 18.1%
base rate to **29,700 pages / 5.1% base rate**, and from 34 to **31** in-universe clients (a
much smaller client loss than an earlier, incorrect estimate — the comparison must be against
clients that actually pass the impression/click floor, not every client present in the
February partition). A 5.1% base rate is a genuine class-imbalance problem for ML-08: accuracy
is meaningless here, and the model will need class weighting or a metric like precision@k /
PR-AUC that doesn't reward always predicting "survived."

#### Limit 3: GSC-only early rows

Clients joining late in the panel have shallow GSC history overall (see `dim_clients.gsc_data_start`
in the warehouse skill). This analysis only uses February and March 2026, so it is less exposed
to this than a longer-history study would be, but any extension of the feature window backward
should re-check per-client history depth before assuming more months are available.

#### Limit 4: The July `dim_content` snapshot approximates February state, unevenly

`word_count`, `search_volume`, and related columns come from a single snapshot dated July 2026 —
four to five months after the February feature window — because the warehouse stores no
historical version of `dim_content`. Measured directly on our universe (Section 3):
- Only **16.8%** of universe pages have `content_updated_date <= 2026-02-28` — for the
  remaining 83.2%, the July snapshot may reflect a post-February edit, not February state.
- `word_count` is NULL for **18.5%** of keyword articles (the dominant content type in-universe)
  and **0%** for feedly/comparison articles — missingness follows `content_type`, so `fillna(0)`
  would inject a content-type signal. We keep `word_count` with a `word_count_missing` flag
  instead.
- `search_volume` is NULL for **100%** of the 175 feedly articles in-universe, 0.7% of keyword
  articles. Same treatment: flag, never impute a shared default across types.

We accept these as bounded, documented approximations rather than dropping the columns
outright — the alternative (no content-shape features at all) is worse — but any feature
importance placed on `word_count` should be read with this caveat attached.

#### Limit 5: `went_dark` doesn't explain why

The label only says a page's clicks fell to zero, not why: a weakened title, a new SERP
feature stealing clicks, a seasonal dip, or a shift in search intent all produce the same
label value. The model sees February aggregates, not query-level detail, snippet text, or
SERP layout. A page flagged `went_dark` needs human review to diagnose the actual cause before
a content team invests in a rewrite — this is a triage tool that identifies pages worth
investigating, not a system that explains or fixes them.

## 5. Output

For each page in the 29,700-page universe (≥100 February impressions, ≥3 February clicks,
published, existing by Feb 28), this analysis emits a predicted probability that the page's
GSC clicks fall to zero in March despite the click history it had in February — delivered as
a ranked triage queue, grouped by client, for the content team to review. Given the 5.1% base
rate, the queue is expected to be most useful evaluated as **precision at the top-k**, not
accuracy. The output identifies pages worth investigating; per Limit 5, it does not diagnose
why a page is declining, and every flagged page needs human review before a rewrite is
commissioned.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.